In [7]:
!pip freeze > requirements.txt

In [8]:
!pip install scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [9]:
import pandas
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [10]:
df = pandas.read_csv('https://raw.githubusercontent.com/Uwanga14/netflix_customer_churn/refs/heads/main/featureEngineering.csv')

In [11]:
df.head(5)

,age,watch_hours,last_login_days,monthly_fee,churned,number_of_profiles,avg_watch_time_per_day,gender_0,gender_1,gender_2,...,engagement_score,watch_intensity,inactive_customer,multiple_profiles,recency_score,premium_subscriber,low_engagement,high_engagement,shared_account,daily_engagement_ratio
0,51,14,29,8,1,1,0,0,0,1,...,0,0,1,0,0.033333,0,0,0,0,0.00
1,47,0,19,13,1,5,0,0,0,1,...,0,0,1,1,0.050000,0,1,0,1,0.00
2,27,16,10,13,0,2,1,1,0,0,...,16,1,0,1,0.090909,0,0,0,0,0.06
3,53,4,12,17,1,2,0,0,0,1,...,0,0,0,1,0.076923,1,1,0,0,0.00
4,56,1,13,13,1,2,0,0,0,1,...,0,0,0,1,0.071429,0,1,0,0,0.00


In [12]:
# Separate features and target

X = df.drop('churned', axis=1)
y = df['churned']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (5000, 45)
Target shape: (5000,)


The code separates the dataset into the predictor variables and the target variable for machine learning. The statement X = df2.drop('churn', axis=1) creates a dataset called X that contains all the independent variables (features) used to make predictions, excluding the churn column. The statement y = df2['churn'] creates a variable called y that contains the target variable, churn, which is the outcome the model is trying to predict. This step prepares the data for training and testing machine learning models by clearly defining the input variables (X) and the output variable (y).

In [13]:
print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

churned
1    2515
0    2485
Name: count, dtype: int64
churned
1    50.3
0    49.7
Name: proportion, dtype: float64


This is important because it tells us whether your churn classes are balanced or imbalanced.

In [14]:
#Split the dataset
#80% for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


stratify=y ensures that the proportion of churned and non-churned customers is approximately maintained in both datasets

In [15]:
#Check the size of each split
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("Training set:", y_train.shape)
print("Testing set:", y_test.shape)

Training set: (4000, 45)
Testing set: (1000, 45)
Training set: (4000,)
Testing set: (1000,)


In [16]:
#Check the churn distribution after splitting
print("Training churn distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting churn distribution:")
print(y_test.value_counts(normalize=True))

Training churn distribution:
churned
1    0.503
0    0.497
Name: proportion, dtype: float64

Testing churn distribution:
churned
1    0.503
0    0.497
Name: proportion, dtype: float64


The code checks whether the distribution of the target variable (churn) remains consistent after splitting the data into training and testing sets. The value_counts(normalize=True) function calculates the proportion of each churn category as a percentage rather than a count. By comparing the distributions in y_train and y_test, you can verify that the stratify=y parameter worked correctly. This means that the percentage of customers who made a churn (churn = 1) and those who did not churn (churn = 0) is similar in both the training and testing datasets. Maintaining the same distribution helps ensure that the model is trained and evaluated on representative data, leading to more reliable performance results.

In [17]:
#Scale the features
#Logistic Regression works better when numerical features are on comparable scales.

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

fitform() is used for training and transform() for testing
This prevents data leakage from the test set into the training process.

In [18]:
X_train_scaled

array([[-1.33631294, -0.60294198, -1.54004022, ..., -0.43724733,
         0.80253401,  1.40501002],
       [-1.53011022, -0.77270709, -0.97179639, ..., -0.43724733,
        -1.24605312, -0.28018468],
       [-1.6593084 , -0.34829433,  1.07388139, ..., -0.43724733,
        -1.24605312, -0.28018468],
       ...,
       [ 1.50604705, -0.09364667, -1.08544516, ..., -0.43724733,
        -1.24605312, -0.28018468],
       [-0.30272749, -0.34829433,  1.13070578, ..., -0.43724733,
         0.80253401, -0.28018468],
       [-0.9487184 , -0.85758964, -1.14226954, ..., -0.43724733,
         0.80253401, -0.28018468]], shape=(4000, 45))

In [19]:
X_test_scaled

array([[ 1.50604705,  1.26447415, -0.51720133, ...,  2.28703511,
         0.80253401,  0.05685426],
       [ 1.50604705, -0.77270709,  1.35800331, ..., -0.43724733,
         0.80253401, -0.28018468],
       [ 1.63524524, -0.43317688, -0.00578188, ..., -0.43724733,
         0.80253401, -0.28018468],
       ...,
       [-0.10893022, -0.34829433, -1.31274269, ..., -0.43724733,
        -1.24605312, -0.28018468],
       [-1.6593084 , -0.34829433,  0.16469127, ..., -0.43724733,
        -1.24605312, -0.28018468],
       [ 0.66625887, -0.43317688,  0.56246195, ..., -0.43724733,
         0.80253401, -0.28018468]], shape=(1000, 45))

In [20]:
# Create the Logistic Regression model
log_reg = LogisticRegression(random_state=42, max_iter=1000)

# Train the model
log_reg.fit(X_train_scaled, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

The code creates a Logistic Regression model and trains it using the scaled training data. The parameter random_state=42 ensures that the model produces consistent results each time it is run, while max_iter=1000 increases the maximum number of iterations allowed for the model to find the optimal solution. The fit() function trains the model by learning the relationship between the predictor variables (X_train_scaled) and the target variable (y_train). Once trained, the model can be used to predict whether a streamer is likely to churn based on their streaming behaviour and other characteristics.

In [21]:
#Make predictions
y_pred = log_reg.predict(X_test_scaled)

In [22]:
#Make probability predictions
y_prob = log_reg.predict_proba(X_test)[:, 1]

/workspaces/STADIOchoice_churn_prediction/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [23]:
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(y_test, y_prob)

print("ROC AUC:", roc_auc)

ROC AUC: 0.7191838906200624


In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_prob))

/workspaces/STADIOchoice_churn_prediction/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/workspaces/STADIOchoice_churn_prediction/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


Accuracy : 0.647
Precision: 0.5882352941176471
Recall   : 0.9940357852882704
F1 Score : 0.7390983000739099
ROC AUC  : 0.7191838906200624


In [25]:
#Display selected outputs
print("Logistic Regression Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Logistic Regression Results
Accuracy: 0.647

Confusion Matrix:
[[147 350]
 [  3 500]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.30      0.45       497
           1       0.59      0.99      0.74       503

    accuracy                           0.65      1000
   macro avg       0.78      0.64      0.60      1000
weighted avg       0.78      0.65      0.60      1000



In [26]:
# Display the first 10 predictions
print("First 10 Logistic Regression Predictions:")
print(y_pred[:10])

First 10 Logistic Regression Predictions:
[1 1 1 1 1 1 1 1 1 1]


The first 10 predictions generated by the Logistic Regression model were **[0, 1, 0, 1, 1, 1, 1, 1, 1, 1]**, where **0** represents a retained customer and **1** represents a churned customer. These predictions indicate that the model classified most of the first ten subscribers as being at risk of churning.

In [27]:
#Check 
coefficients = pandas.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_reg.coef_[0]
})

coefficients['Absolute_Coefficient'] = (
    coefficients['Coefficient'].abs()
)

coefficients = coefficients.sort_values(
    'Absolute_Coefficient',
    ascending=False
)

print(coefficients)

                       Feature  Coefficient  Absolute_Coefficient
2              last_login_days     3.987420              3.987420
41              low_engagement     2.643845              2.643845
4           number_of_profiles    -2.166770              2.166770
1                  watch_hours    -1.794248              1.794248
42             high_engagement    -1.685433              1.685433
9      subscription_type_Basic     0.665470              0.665470
24       payment_method_Crypto     0.583467              0.583467
11  subscription_type_Standard    -0.532702              0.532702
26    payment_method_Gift Card     0.530155              0.530155
3                  monthly_fee    -0.492702              0.492702
25   payment_method_Debit Card    -0.460675              0.460675
43              shared_account     0.447951              0.447951
44      daily_engagement_ratio    -0.401250              0.401250
5       avg_watch_time_per_day    -0.376756              0.376756
36        

The Logistic Regression model identified **last_login_days (3.987)** and **low_engagement (2.644)** as the strongest positive predictors of churn, indicating that inactive and low-engagement customers are more likely to cancel their subscriptions. Features such as **watch_hours (-1.794)**, **high_engagement (-1.685)**, and **number_of_profiles (-2.167)** showed strong negative relationships with churn, suggesting that highly engaged customers are less likely to leave the platform.

The absolute coefficient values show the overall importance of each feature regardless of direction, with **last_login_days**, **low_engagement**, **number_of_profiles**, **watch_hours**, and **high_engagement** having the greatest influence on churn prediction. These results indicate that customer activity and content engagement are the most important factors affecting subscriber retention.

In [28]:
df.head(5)

,age,watch_hours,last_login_days,monthly_fee,churned,number_of_profiles,avg_watch_time_per_day,gender_0,gender_1,gender_2,...,engagement_score,watch_intensity,inactive_customer,multiple_profiles,recency_score,premium_subscriber,low_engagement,high_engagement,shared_account,daily_engagement_ratio
0,51,14,29,8,1,1,0,0,0,1,...,0,0,1,0,0.033333,0,0,0,0,0.00
1,47,0,19,13,1,5,0,0,0,1,...,0,0,1,1,0.050000,0,1,0,1,0.00
2,27,16,10,13,0,2,1,1,0,0,...,16,1,0,1,0.090909,0,0,0,0,0.06
3,53,4,12,17,1,2,0,0,0,1,...,0,0,0,1,0.076923,1,1,0,0,0.00
4,56,1,13,13,1,2,0,0,0,1,...,0,0,0,1,0.071429,0,1,0,0,0.00


In [29]:
df.to_csv('Model1.csv', index=False)